<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/single_prompt_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series - Example: Single Prompt Activation Analysis
![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)
This notebook demonstrates how to use [OptiPFair](https://github.com/peremartra/optipfair) to capture and visualize internal activations for a single text prompt.
It uses the public bias API to inspect what happens inside MLP layers (gate projection, up projection and down projection input) when a model processes a given query.

## Recommended Environment
- **Platform**: [Google Colab](https://colab.research.google.com)
- **Hardware**: GPU T4 or higher (CPU also supported for small models)
- **Dependencies**: Installed in Section 0

## by Pere Martra
- [LinkedIn](https://www.linkedin.com/in/pere-martra)
- [GitHub](https://github.com/peremartra)
- [X / Twitter](https://x.com/peremartra)

---
> If you find this useful, please star the [repository](https://github.com/peremartra/optipfair).
---
If you want your favorite LLM to create code with optipfair, provide [optipfair_llm_reference_manual.txt](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt).

## 0. Environment and Dependencies

In [ ]:
!pip install --upgrade git+https://github.com/peremartra/optipfair.git
#!pip install -q optipfair==0.4.0
!pip install -q transformers torch matplotlib

## 1. Global Configuration

In [ ]:
import torch

# Model
MODEL_ID = "meta-llama/Llama-3.2-1B"

# Prompt to analyze
PROMPT = "What's the most significant building in Barcelona?"

# Target layer types to capture.
# NOTE: "down_proj_input" is opt-in-only and must be listed explicitly.
TARGET_LAYERS = ["gate_proj", "up_proj", "down_proj_input"]

# Layer keys to visualize (one heatmap per key).
# Adjust the layer index to inspect a different transformer block.
LAYER_INDEX = 0
VIZ_LAYER_KEYS = [
    f"gate_proj_layer_{LAYER_INDEX}",
    f"up_proj_layer_{LAYER_INDEX}",
    f"down_proj_input_layer_{LAYER_INDEX}",
]

# Visualization parameters
BIN_SIZE = 64
CMAP = "YlOrRd"
VMAX_PERCENTILE = 99.0

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
model = model.to(DEVICE)
model.eval()

print("Model loaded successfully.")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Capture Single-Prompt Activations

We use `get_prompt_activations` to run the model on a single prompt and capture
the internal activations for the layer types listed in `TARGET_LAYERS`.

The result is a dictionary `{layer_key: tensor}` where each tensor has shape
`(batch, seq_len, hidden_dim)`.

In [ ]:
from optipfair.bias import get_prompt_activations

print(f"Prompt: {PROMPT!r}")
print(f"Capturing activations for layer types: {TARGET_LAYERS}")

activations = get_prompt_activations(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    target_layers=TARGET_LAYERS,
)

print(f"\nCaptured {len(activations)} activation tensors:")
for key, tensor in sorted(activations.items()):
    print(f"  {key}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

## 4. Visualize Activation Heatmaps

Each heatmap shows:
- **Y-axis (rows)**: token positions in the prompt.
- **X-axis (columns)**: neuron bins (each bin groups `BIN_SIZE` consecutive neurons).

We plot three projections from the same transformer block to compare how
the gate, up, and down_proj_input activations differ for the same prompt.

In [ ]:
from optipfair.bias import visualize_prompt_heatmap

# --- Gate Projection ---
print(f"Heatmap: {VIZ_LAYER_KEYS[0]}")
visualize_prompt_heatmap(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    layer_key=VIZ_LAYER_KEYS[0],
    bin_size=BIN_SIZE,
    cmap=CMAP,
    vmax_percentile=VMAX_PERCENTILE,
    show=True,
)

In [ ]:
# --- Up Projection ---
print(f"Heatmap: {VIZ_LAYER_KEYS[1]}")
visualize_prompt_heatmap(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    layer_key=VIZ_LAYER_KEYS[1],
    bin_size=BIN_SIZE,
    cmap=CMAP,
    vmax_percentile=VMAX_PERCENTILE,
    show=True,
)

In [ ]:
# --- Down Projection Input ---
# NOTE: down_proj_input captures the tensor entering the down projection
# (i.e., gate * up after the SiLU activation). It is opt-in-only and was
# explicitly requested via TARGET_LAYERS above.
print(f"Heatmap: {VIZ_LAYER_KEYS[2]}")
visualize_prompt_heatmap(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    layer_key=VIZ_LAYER_KEYS[2],
    bin_size=BIN_SIZE,
    cmap=CMAP,
    vmax_percentile=VMAX_PERCENTILE,
    show=True,
)

## 5. Summary and Next Steps

In this notebook we:
1. Loaded `meta-llama/Llama-3.2-1B` using the Hugging Face transformers library.
2. Captured internal activations for the prompt _"What's the most significant building in Barcelona?"_
   using `get_prompt_activations` with explicit `target_layers`.
3. Visualized three MLP projections (gate, up, down_proj_input) as token × neuron-bin heatmaps.

### Possible extensions

- **Change the layer index**: set `LAYER_INDEX` to a higher block (e.g. `8` or `15`) to inspect deeper representations.
- **Compare two prompts**: use `get_activation_pairs` + `visualize_heatmap` to measure the difference between two demographically varied prompts.
- **Save figures to disk**: pass `output_dir="./heatmaps"` to `visualize_prompt_heatmap` to persist the plots.
- **Quantify bias**: use the metrics module to compute numerical bias scores across layer types.